# GT(gt_schoolnames.csv)와 대조 — school_candidate가 실제 학교명인지 검증

`count.ipynb` -> `preprocessing2.ipynb` 순서로 먼저 실행해서 만든
`data/processed/preprocessed_school_candidate.csv`(`preprocessed_candidate` 컬럼 포함)를 입력으로 사용.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [2]:
import pandas as pd

df = pd.read_csv(
    REPO_ROOT / "data" / "processed" / "preprocessed_school_candidate.csv", encoding="utf-8-sig"
)
# 후보 0건인 행은 NaN으로 읽히므로 방어
df["school_candidate"] = df["school_candidate"].fillna("")
df["preprocessed_candidate"] = df["preprocessed_candidate"].fillna("")
df.shape

(1000, 7)

## preprocessed_candidate가 GT 정식명 목록에 실제로 있는지만 확인

In [3]:
gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_names = set(gt_df["학교명"])


def gt_match(expanded_text: str) -> str:
    """preprocessed_candidate는 이미 preprocessing2.ipynb에서 약어->정식명, 접미사 확장까지
    끝낸 상태라, 여기선 GT 정식명 목록에 실제로 있는지만 확인하면 됨."""
    return " ".join(tok for tok in expanded_text.split() if tok in gt_names)


df["gt_match"] = df["preprocessed_candidate"].apply(gt_match)
df["gt_match_count"] = df["gt_match"].apply(lambda s: len(s.split()) if s else 0)
df[["comment", "school_candidate", "preprocessed_candidate", "gt_match", "gt_match_count"]].head(20)

,comment,school_candidate,preprocessed_candidate,gt_match,gt_match_count
0,동국대학교,동국대학교,동국대학교,동국대학교,1
1,이번엔 중동고 차례입니다 🍗,중동고,중동고등학교,중동고등학교,1
2,우리 반/동아리 대표로 서초초 신청합니다!,서초초,서초초등학교,,0
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 건국대,잠실중학교 건국대학교,잠실중학교 건국대학교,2
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 서초초 보고,이화여대부속초등학교 서초초등학교 보고등학교,,0
5,학식 말고 치킨 먹고 싶어요 인하대학교,인하대학교,인하대학교,인하대학교,1
6,서울공업고.. 오늘만 기다렸어요,서울공업고,서울공업고등학교,서울공업고등학교,1
7,서강대 학생들 모여라,서강대,서강대학교,서강대학교,1
8,동아리방에서 기다릴게요 서초중,서초중,서초중학교,서초중학교,1
9,대치중 학생입니다 대치중 뽑아주세요,대치중,대치중학교,대치중학교,1


In [ ]:
# gt_match 중 실제 매칭된 것만 최종 정답(ans) 컬럼으로 따로 저장 + 옆에 개수(ans_count)
df["ans"] = df["gt_match"]
df["ans_count"] = df["gt_match_count"]
df[["comment", "ans", "ans_count","preprocessed_candidate"]].head(20)

,comment,ans,ans_count
0,동국대학교,동국대학교,1
1,이번엔 중동고 차례입니다 🍗,중동고등학교,1
2,우리 반/동아리 대표로 서초초 신청합니다!,,0
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중학교 건국대학교,2
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,,0
5,학식 말고 치킨 먹고 싶어요 인하대학교,인하대학교,1
6,서울공업고.. 오늘만 기다렸어요,서울공업고등학교,1
7,서강대 학생들 모여라,서강대학교,1
8,동아리방에서 기다릴게요 서초중,서초중학교,1
9,대치중 학생입니다 대치중 뽑아주세요,대치중학교,1


In [5]:
# school_candidate_count 대비 실제 GT에 매칭된 개수 분포
# (예: count=1인데 gt_match_count=0 -> "연대"처럼 약칭이라 GT엔 없는 케이스)
df["gt_match_count"].value_counts().sort_index()

gt_match_count
0    383
1    590
2     27
Name: count, dtype: int64

## GT 매칭 실패 케이스 — 약어 후보 vs 오검출(일반 명사) 구분 필요

In [6]:
out_path = REPO_ROOT / "data" / "processed" / "gt_match_results.csv"
df[
    [
        "comment_id",
        "comment",
        "comment_noun",
        "school_candidate",
        "preprocessed_candidate",
        "school_candidate_count",
        "gt_match",
        "gt_match_count",
        "ans",
        "ans_count",
    ]
].to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

WindowsPath('d:/Study/dongguk_university/dreampath/data/processed/gt_match_results.csv')

In [7]:
out_path = REPO_ROOT / "data" / "processed" / "gt_match_results.csv"
df[
    [
        "comment_id",
        "comment",
        "comment_noun",
        "school_candidate",
        "preprocessed_candidate",
        "school_candidate_count",
        "gt_match",
        "gt_match_count",
    ]
].to_csv(out_path, index=False, encoding="utf-8-sig")
out_path

WindowsPath('d:/Study/dongguk_university/dreampath/data/processed/gt_match_results.csv')